In [2]:
import shutil
import logging
import hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime

class BronzePipeline:
    def __init__(self, base_dir_path: str, expected_files: list):
        self.base_dir = Path(base_dir_path)
        self.raw_dir = self.base_dir / 'raw'
        self.lakehouse_dir = self.base_dir / 'lakehouse'
        self.bronze_dir = self.lakehouse_dir / 'bronze'

        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.current_run_dir = self.bronze_dir / self.timestamp

        # Reset manifest records per pipeline instantiation
        self.manifest_records = []
        self.expected_files = expected_files

        self._setup_directories()

    def _setup_directories(self):
        self.raw_dir.mkdir(parents=True, exist_ok=True)
        self.current_run_dir.mkdir(parents=True, exist_ok=True)

    def calculate_md5(self, file_path: Path) -> str:
        hash_md5 = hashlib.md5()
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()

    def validate_file(self, file_path: Path):
        if not file_path.exists():
            return False, "FILE_NOT_FOUND"
        if file_path.stat().st_size == 0:
            return False, "EMPTY_FILE"
        return True, "VALID"

    def ingest_file(self, source_file: Path, destination_file: Path):
        source_hash = self.calculate_md5(source_file)
        shutil.copy2(source_file, destination_file)
        destination_hash = self.calculate_md5(destination_file)

        if source_hash != destination_hash:
            raise Exception("CHECKSUM_MISMATCH")
        return source_hash

    def create_manifest(self):
        if not self.manifest_records:
            logging.warning("No records to write to manifest.")
            return

        manifest_df = pd.DataFrame(self.manifest_records)
        manifest_path = self.current_run_dir / 'ingestion_manifest.csv'
        manifest_df.to_csv(manifest_path, index=False)
        logging.info(f"Manifest created: {manifest_path}")

    def run(self):
        logging.info("========== BRONZE LAYER STARTED ==========")

        for file_name in self.expected_files:
            source_file = self.raw_dir / file_name
            destination_file = self.current_run_dir / file_name

            record = {
                "file_name": file_name,
                "ingestion_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "status": None,
                "md5_checksum": None,
                "file_size_mb": None,
                "reason": None
            }

            is_valid, validation_message = self.validate_file(source_file)

            if not is_valid:
                logging.error(f"{file_name} -> {validation_message}")
                record.update({"status": "FAILED", "reason": validation_message})
                self.manifest_records.append(record)
                continue

            try:
                checksum = self.ingest_file(source_file, destination_file)
                file_size_mb = round(source_file.stat().st_size / (1024 * 1024), 2)

                logging.info(f"SUCCESS -> {file_name} | MD5={checksum[:8]}... | SIZE={file_size_mb} MB")
                record.update({
                    "status": "SUCCESS",
                    "md5_checksum": checksum,
                    "file_size_mb": file_size_mb
                })

            except Exception as e:
                logging.error(f"{file_name} -> {str(e)}")
                record.update({"status": "FAILED", "reason": str(e)})

            self.manifest_records.append(record)

        self.create_manifest()
        logging.info("========== BRONZE LAYER COMPLETED ==========")

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s', force=True)

    # Colab setup
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount('/content/drive', force_remount=True)
    except ImportError:
        pass

    # Configuration
    BASE_DIR = '/content/drive/MyDrive/DataStorm_TeamZypher'
    EXPECTED_FILES = [
        "transactions_history_final.csv",
        "outlet_master.csv",
        "outlet_coordinates.csv",
        "holiday_list.csv",
        "distributor_seasonality_details.csv"
    ]

    # Execute
    bronze_pipeline = BronzePipeline(base_dir_path=BASE_DIR, expected_files=EXPECTED_FILES)
    bronze_pipeline.run()

2026-05-16 22:58:59,298 - [INFO] - ========== BRONZE LAYER STARTED ==========
2026-05-16 22:59:01,451 - [INFO] - SUCCESS -> transactions_history_final.csv | MD5=68342a3e... | SIZE=161.32 MB
2026-05-16 22:59:02,028 - [INFO] - SUCCESS -> outlet_master.csv | MD5=96de9d03... | SIZE=0.48 MB
2026-05-16 22:59:02,068 - [INFO] - SUCCESS -> outlet_coordinates.csv | MD5=a475e1ee... | SIZE=0.55 MB
2026-05-16 22:59:02,092 - [INFO] - SUCCESS -> holiday_list.csv | MD5=c59fa68c... | SIZE=0.02 MB
2026-05-16 22:59:02,114 - [INFO] - SUCCESS -> distributor_seasonality_details.csv | MD5=530d7f7a... | SIZE=0.01 MB
2026-05-16 22:59:02,127 - [INFO] - Manifest created: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/bronze/20260516_225859/ingestion_manifest.csv
2026-05-16 22:59:02,128 - [INFO] - ========== BRONZE LAYER COMPLETED ==========
